# 모델 구축 구조

In [ ]:
import pandas as pd
df = pd.read_csv('Total_Data_v2.csv')
head(df)

✅ 종속변수 설정

“위기예측점수(risk_score)” 또는 “매출 안정성(변동성) 여부”, “급감여부(-20% YoY)” 등을 기준으로
위기(1) / 안정(0) 이진 레이블 생성

In [ ]:
df['위기여부'] = df['급감여부(-20% YoY)'].apply(lambda x: 1 if x == 'Y' else 0)


In [ ]:
# 결측치 처
df = df.fillna(df.median(numeric_only=True))


In [ ]:
# 범주형 변수 처리
df = pd.get_dummies(df, columns=['가맹점지역', '업종', '상권', '브랜드구분코드'])


2단계: 데이터 분할 및 정규화/스케일링
✅ 독립변수(X), 종속변수(y) 정의

In [ ]:
X = df[['취소율 구간', '배달매출금액 비율', '매출 안정성(변동성) CV(3개월)', '전월대비 매출금액 감소율(%)',
         '3개월 연속 감소 여부', '6개월 하락추세 여부', '매출금액 구간_회복지수_6개월',
         '동일 업종 매출금액 비율_x', '동일 상권 내 매출 순위 비율_x',
         '동일 업종 내 해지 가맹점 비중_x', '동일 상권 내 해지 가맹점 비중',
         '재방문 고객 비중_x', '신규 고객 비중', '고객분포_다양성지수',
         '거주 이용 고객 비율', '직장 이용 고객 비율', '유동인구 이용 고객 비율']]
y = df['위기여부']


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# 정규화/표준화
from sklearn.preprocessing import MinMaxScaler, StandardScaler

minmax = MinMaxScaler()
std = StandardScaler()

X_train_minmax = minmax.fit_transform(X_train)
X_test_minmax = minmax.transform(X_test)

X_train_std = std.fit_transform(X_train)
X_test_std = std.transform(X_test)


3단계: 예측 모델 구축 및 Grid Search
✅ RandomForest 하이퍼파라미터 탐색

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='recall')
grid_rf.fit(X_train_std, y_train)
best_rf = grid_rf.best_estimator_


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': best_rf,
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}


4단계: 모델 평가 및 비교

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

for name, model in models.items():
    model.fit(X_train_std, y_train)
    y_pred = model.predict(X_test_std)
    y_prob = model.predict_proba(X_test_std)[:,1]

    print(f'📈 {name}')
    print(f'Accuracy : {accuracy_score(y_test, y_pred):.3f}')
    print(f'Precision: {precision_score(y_test, y_pred):.3f}')
    print(f'Recall   : {recall_score(y_test, y_pred):.3f}')  # 중요
    print(f'F1 Score : {f1_score(y_test, y_pred):.3f}')
    print(f'AUC-ROC  : {roc_auc_score(y_test, y_prob):.3f}')
    print('-'*40)


# 클로드

In [ ]:
"""
2025 빅콘테스트: 가맹점 위기 예측 AI 조기 경보 시스템
전체 분석 파이프라인
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글 폰트 설정
plt.rcParams['axes.unicode_minus'] = False

# 전처리 및 스케일링
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer

# 모델
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# 평가 지표
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)

# ============================================================================
# 1단계: 데이터 불러오기 및 전처리
# ============================================================================

print("="*80)
print("1단계: 데이터 불러오기 및 전처리")
print("="*80)

# 1-1. 데이터 불러오기
df = pd.read_csv('/content/drive/MyDrive/GamjaNeverDie/Total_Data/Total_Data_v2.csv', encoding='utf-8-sig')
print(f"\n✅ 데이터 로드 완료: {df.shape[0]:,}개 행, {df.shape[1]:,}개 열")
print(f"\n📊 데이터 기본 정보:")
print(df.info())

# 1-2. 종속변수(Y) 생성
# 방법 1: 기존 컬럼 활용 (폐업/휴업 여부가 있는 경우)
if '폐업여부' in df.columns:
    df['위기여부'] = df['폐업여부'].replace([np.inf, -np.inf], np.nan).fillna(0).astype(int)
    print("\n✅ 기존 '폐업여부' 컬럼을 종속변수로 사용")

# 방법 2: 위기 점수 계산 방식 (복합 지표 활용)
else:
    print("\n📊 위기 점수 계산 중...")

    # 위기 신호 지표들 (각 지표를 0~1로 정규화하여 합산)
    crisis_indicators = []

    # 매출 관련 위기 신호
    if '전월대비 매출금액 감소율(%)' in df.columns:
        temp = df['전월대비 매출금액 감소율(%)'].replace([np.inf, -np.inf], np.nan).fillna(0)
        crisis_indicators.append((temp < -10).astype(int))

    if '3개월 연속 감소 여부' in df.columns:
        temp = df['3개월 연속 감소 여부'].replace([np.inf, -np.inf], np.nan).fillna(0)
        crisis_indicators.append(temp.astype(int))

    if '급감여부(-20% YoY)' in df.columns:
        temp = df['급감여부(-20% YoY)'].replace([np.inf, -np.inf], np.nan).fillna(0)
        crisis_indicators.append(temp.astype(int))

    # 거래 안정성 관련
    if '매출 안정성(변동성) 여부' in df.columns:
        temp = df['매출 안정성(변동성) 여부'].replace([np.inf, -np.inf], np.nan).fillna(0)
        crisis_indicators.append(temp.astype(int))

    # 위기 점수 계산 (0~1 범위)
    if crisis_indicators:
        crisis_score = sum(crisis_indicators) / len(crisis_indicators)
        # 임계값 설정 (상위 30%를 위기로 분류)
        threshold = crisis_score.quantile(0.7)
        df['위기여부'] = (crisis_score >= threshold).astype(int)
        print(f"✅ 위기 점수 임계값: {threshold:.3f}")
    else:
        # 기본값: 매출 감소율 기준
        print("⚠️ 기본 지표로 위기여부 생성 (매출 감소율 -20% 이하)")
        df['위기여부'] = 0  # 임시값

print(f"\n📊 종속변수 분포:")
print(df['위기여부'].value_counts())
print(f"위기 비율: {df['위기여부'].mean()*100:.2f}%")

# 1-3. 독립변수(X) 선택
# 사용자가 제공한 주요 변수들을 기본으로 설정
feature_candidates = [
    # 거래 안정성
    '매출 안정성(변동성) CV(3개월)', '매출 안정성(변동성) 여부',
    '배달매출금액 비율', '매출탄력도(6개월)', '고객분포_다양성지수',

    # 매출
    '전월대비 매출금액 감소율(%)', '3개월 연속 감소 여부', '6개월 하락추세 여부',
    '매출금액 구간_회복지수_6개월', '급감여부(-20% YoY)', '신규 고객 비중',
    '매출건수 백분위', '유니크 고객 수 백분위',

    # 상대적 평가
    '동일 업종 내 매출 순위 비율', '동일 상권 내 매출 순위 비율',
    '동일 업종 내 해지 가맹점 비중', '동일 상권 내 해지 가맹점 비중',

    # 고객 특성
    '재방문 고객 비중', '남성 30대 고객 비중', '여성 30대 고객 비중',
    '거주 이용 고객 비율', '직장 이용 고객 비율', '유동인구 이용 고객 비율'
]

# 실제 존재하는 컬럼만 선택
X_features = [col for col in feature_candidates if col in df.columns]
print(f"\n✅ 선택된 독립변수: {len(X_features)}개")
print(X_features[:10], "...")  # 처음 10개만 출력

# 1-4. 결측치 및 이상치 처리
print("\n📊 결측치 확인:")
missing_summary = df[X_features + ['위기여부']].isnull().sum()
missing_summary = missing_summary[missing_summary > 0]
if len(missing_summary) > 0:
    print(missing_summary)
    print(f"\n✅ 결측치 처리 중... (수치형: 중앙값, 범주형: 최빈값)")
else:
    print("결측치 없음")

# 수치형과 범주형 분리
numeric_features = df[X_features].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[X_features].select_dtypes(exclude=[np.number]).columns.tolist()

# 결측치 처리
if numeric_features:
    num_imputer = SimpleImputer(strategy='median')
    df[numeric_features] = num_imputer.fit_transform(df[numeric_features])

if categorical_features:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[categorical_features] = cat_imputer.fit_transform(df[categorical_features])
    # One-Hot Encoding
    df = pd.get_dummies(df, columns=categorical_features, drop_first=True)
    # 새로 생성된 더미 변수를 X_features에 추가
    new_dummy_cols = [col for col in df.columns if any(cat in col for cat in categorical_features)]
    X_features = [col for col in X_features if col not in categorical_features] + new_dummy_cols

print(f"✅ 전처리 완료: 최종 독립변수 {len(X_features)}개")

# ============================================================================
# 2단계: 데이터 분할 및 정규화/스케일링
# ============================================================================

print("\n" + "="*80)
print("2단계: 데이터 분할 및 정규화/스케일링")
print("="*80)

# 2-1. 데이터 분할
X = df[X_features]
y = df['위기여부']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n✅ 데이터 분할 완료 (Train: {len(X_train):,}, Test: {len(X_test):,})")
print(f"   Train 위기 비율: {y_train.mean()*100:.2f}%")
print(f"   Test 위기 비율: {y_test.mean()*100:.2f}%")

# 2-2. 스케일링 객체 생성
scaler_minmax = MinMaxScaler()
scaler_standard = StandardScaler()

# Min-Max Normalization
X_train_minmax = scaler_minmax.fit_transform(X_train)
X_test_minmax = scaler_minmax.transform(X_test)
print("\n✅ Min-Max Normalization 완료")

# Standardization
X_train_standard = scaler_standard.fit_transform(X_train)
X_test_standard = scaler_standard.transform(X_test)
print("✅ Standardization 완료")

# 2-3. Grid Search용 파라미터 설정
param_grids = {
    'LogisticRegression': {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l2'],
        'max_iter': [1000]
    },
    'RandomForest': {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1],
        'subsample': [0.8, 1.0]
    }
}

print("\n✅ Grid Search 파라미터 설정 완료")

# ============================================================================
# 3단계: 예측 모델 구축 및 검증 (Standardization 기준)
# ============================================================================

print("\n" + "="*80)
print("3단계: 예측 모델 구축 (Standardization + Grid Search)")
print("="*80)

models = {
    'LogisticRegression': LogisticRegression(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

best_models = {}
cv_results = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"🔍 {name} - Grid Search 진행 중...")
    print(f"{'='*60}")

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=5,
        scoring='recall',  # 위기 예측에서는 Recall이 중요
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train_standard, y_train)

    best_models[name] = grid_search.best_estimator_
    cv_results[name] = {
        'best_params': grid_search.best_params_,
        'best_score': grid_search.best_score_
    }

    print(f"✅ 최적 파라미터: {grid_search.best_params_}")
    print(f"✅ CV Recall 점수: {grid_search.best_score_:.4f}")

# ============================================================================
# 4단계: 모델 평가 및 비교
# ============================================================================

print("\n" + "="*80)
print("4단계: 모델 평가 및 비교")
print("="*80)

results = []

for name, model in best_models.items():
    print(f"\n{'='*60}")
    print(f"📊 {name} 평가")
    print(f"{'='*60}")

    # 예측
    y_pred = model.predict(X_test_standard)
    y_pred_proba = model.predict_proba(X_test_standard)[:, 1]

    # 성능 지표 계산
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_pred_proba)

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC-ROC': auc_roc
    })

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f} ⭐ (위기 예측 핵심 지표)")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC-ROC:   {auc_roc:.4f}")

    print(f"\n혼동 행렬:")
    cm = confusion_matrix(y_test, y_pred)
    print(cm)
    print(f"\n분류 리포트:")
    print(classification_report(y_test, y_pred, target_names=['안정', '위기']))

# 결과 비교 테이블
print("\n" + "="*80)
print("📊 모델 성능 종합 비교")
print("="*80)

results_df = pd.DataFrame(results).sort_values('Recall', ascending=False)
print(results_df.to_string(index=False))

# 최고 성능 모델 추천
best_model_name = results_df.iloc[0]['Model']
best_recall = results_df.iloc[0]['Recall']

print("\n" + "="*80)
print("🏆 최종 추천 모델")
print("="*80)
print(f"모델: {best_model_name}")
print(f"Recall: {best_recall:.4f}")
print(f"\n💡 가맹점 위기 예측에서는 Recall(재현율)이 가장 중요합니다.")
print(f"   실제 위기 상황을 놓치지 않고 감지하는 것이 핵심이기 때문입니다.")

# 변수 중요도 시각화 (RandomForest 또는 XGBoost)
if best_model_name in ['RandomForest', 'XGBoost']:
    print("\n" + "="*80)
    print("📊 변수 중요도 분석")
    print("="*80)

    feature_importance = pd.DataFrame({
        'Feature': X_features,
        'Importance': best_models[best_model_name].feature_importances_
    }).sort_values('Importance', ascending=False)

    print("\nTop 10 중요 변수:")
    print(feature_importance.head(10).to_string(index=False))

print("\n" + "="*80)
print("✅ 분석 완료!")
print("="*80)

1단계: 데이터 불러오기 및 전처리

✅ 데이터 로드 완료: 86,590개 행, 53개 열

📊 데이터 기본 정보:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86590 entries, 0 to 86589
Data columns (total 53 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           86590 non-null  int64  
 1   가맹점구분번호              86590 non-null  object 
 2   기준년월                 86590 non-null  object 
 3   가맹점 운영개월수 구간         86590 non-null  object 
 4   매출금액 구간              86590 non-null  object 
 5   매출건수 구간              86590 non-null  object 
 6   유니크 고객 수 구간          86590 non-null  object 
 7   객단가 구간               86590 non-null  object 
 8   취소율 구간               79958 non-null  object 
 9   배달매출금액 비율            86590 non-null  float64
 10  동일 업종 매출금액 비율_x      86590 non-null  float64
 11  동일 업종 매출건수 비율        86590 non-null  float64
 12  동일 업종 내 매출 순위 비율     86590 non-null  float64
 13  동일 상권 내 매출 순위 비율_x   86590 non-null  float64
 14  동일 업종 내 해지 가맹점 비중_x  

In [ ]:
# run_pipeline.py
# 우리 동네 가맹점 위기 예측 파이프라인 (프롬프트 1번 요구사항 구현)

import os
from pathlib import Path
import numpy as np
import pandas as pd

# =========================
# 0) CONFIG
# =========================
CSV_PATH = Path(r"C:/Users/SSAFY/Desktop/Total_Data_v2.csv")
RANDOM_STATE = 42
TEST_SIZE = 0.2
MAX_ROWS_FOR_SPEED = 6000     # 전체 데이터 쓰려면 None 또는 큰 값으로
TARGET_QUANTILE_FOR_RISK = 0.30   # 점수형 타깃 하위 30%를 '위기(1)'
DERIVED_RISK_TOP = 0.70           # 프록시 위험 점수 상위 30%를 '위기(1)'

# 결과 저장 경로
OUT_METRICS = Path("./model_metrics.csv")
OUT_SUMMARY = Path("./dataset_target_summary.csv")
OUT_FEATURES = Path("./used_features.csv")

# 프롬프트 표 기반 키워드(부분 일치)
FEATURE_KEYWORDS = {
    "거래안정성": ["취소", "거래", "안정", "변동성", "cv", "c.v"],
    "매출패턴" : ["매출", "전월대비", "연속", "하락", "하락추세", "회복지수", "급감", "yoy", "건수"],
    "상대평가" : ["동일 업종", "동일 상권", "순위", "하위", "비율"],
    "고객특성" : ["재방문", "신규", "남성", "여성", "거주", "직장", "유동인구", "다양성"],
    "배달"    : ["배달"],
}
# 파생 위험 점수 계산에 우선 반영할 키워드
RISKY_WORDS = ["감소","하락","급감","연속","순위","취소","위기","리스크","risk"]

# =========================
# 1) 데이터 불러오기 & 전처리
# =========================
def load_csv(path: Path, sep=None):
    # 1) 존재 확인
    if not path.exists():
        raise FileNotFoundError(f"CSV 파일을 못 찾았어: {path.resolve()}")

    # 2) 파일 오픈 가능 여부(락/권한) 점검
    try:
        with open(path, "rb") as f:
            _ = f.read(1024)
    except Exception as e:
        raise RuntimeError(f"CSV 파일을 열 수가 없어(권한/잠금/경로 문제일 수 있음): {e}")

    # 3) 구분자 후보
    seps_to_try = [sep] if sep else [",", "\t", ";", "|"]

    # 4) 인코딩 후보
    encodings_to_try = ["utf-8-sig", "cp949", "ms949", "euc-kr", "utf-8", "latin1"]

    last_err = None
    for enc in encodings_to_try:
        for sp in seps_to_try:
            # 먼저 python 엔진 (on_bad_lines 사용 가능)
            try:
                df = pd.read_csv(
                    path,
                    encoding=enc,
                    sep=sp,
                    engine="python",
                    on_bad_lines="skip"  # 깨진 라인 스킵
                )
                if df.shape[1] < 1:
                    raise ValueError("열이 1개 미만으로 읽혔어. 구분자/인코딩이 안 맞을 수 있어.")
                print(f"[INFO] read_csv OK: engine='python', encoding='{enc}', sep='{sp}', shape={df.shape}")
                return df
            except Exception as e:
                last_err = e
                # python 엔진 실패 시 C 엔진으로 재시도 (on_bad_lines 미지원)
                try:
                    df = pd.read_csv(
                        path,
                        encoding=enc,
                        sep=sp,
                        engine="c"
                        # low_memory 옵션은 기본값(True)로 두자. C엔진에서만 의미 있음.
                    )
                    if df.shape[1] < 1:
                        raise ValueError("열이 1개 미만으로 읽혔어. 구분자/인코딩이 안 맞을 수 있어.")
                    print(f"[INFO] read_csv OK: engine='c', encoding='{enc}', sep='{sp}', shape={df.shape}")
                    return df
                except Exception as e2:
                    last_err = e2
                    continue

    raise RuntimeError(
        "CSV 로드 실패: 경로/잠금/구분자/인코딩을 확인해줘. "
        f"마지막 에러: {type(last_err).__name__}: {last_err}"
    )


def is_binary_series(s: pd.Series) -> bool:
    vals = pd.Series(s).dropna().unique()
    if len(vals) <= 2:
        return True
    canon = {str(x).strip().lower() for x in vals}
    return canon <= {"0","1","y","n","yes","no","true","false"}

def pick_features(df: pd.DataFrame, exclude=set()):
    lower = {c: str(c).lower() for c in df.columns}
    selected = set()
    for c in df.columns:
        name_low = lower[c]
        for keys in FEATURE_KEYWORDS.values():
            if any(k.lower() in name_low for k in keys):
                if c not in exclude:
                    selected.add(c)
                break
    # 수치형만
    selected = [c for c in selected if pd.api.types.is_numeric_dtype(df[c])]
    # 부족하면 결측률 낮은 수치형 보충
    if len(selected) < 5:
        num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c not in exclude]
        na_rate = {c: df[c].isna().mean() for c in num}
        for c in sorted(num, key=lambda x: na_rate[x])[:20]:
            if c not in selected: selected.append(c)
    return selected

def build_target(df: pd.DataFrame):
    lower = {c: str(c).lower() for c in df.columns}
    bin_col, score_col = None, None

    # (1) 위기/폐업/해지/휴업 이진
    for c in df.columns:
        nm = lower[c]
        if any(k in nm for k in ["위기","폐업","해지","휴업"]) and is_binary_series(df[c]):
            bin_col = c; break

    if bin_col is not None:
        y_map = {"y":1,"yes":1,"true":1,"1":1,"n":0,"no":0,"false":0,"0":0}
        def to01(v):
            if pd.isna(v): return np.nan
            s = str(v).strip().lower()
            if s in y_map: return y_map[s]
            try:
                f = float(s)
                return 1 if f==1 else (0 if f==0 else np.nan)
            except: return np.nan
        y = df[bin_col].map(to01).astype('float')
        return y, f"binary:{bin_col}", bin_col, None, None

    # (2) score/risk/점수 수치 → 하위 30% = 위기
    for c in df.columns:
        nm = lower[c]
        if any(k in nm for k in ["리스크","risk","score","점수"]) and pd.api.types.is_numeric_dtype(df[c]):
            score_col = c; break
    if score_col is not None:
        s = pd.to_numeric(df[score_col], errors="coerce")
        thr = s.quantile(TARGET_QUANTILE_FOR_RISK)
        y = (s <= thr).astype(int)
        return y, f"score:{score_col}<=p{int(TARGET_QUANTILE_FOR_RISK*100)}({thr:.4f})", None, score_col, float(thr)

    # (3) 파생 위험 점수: 위험 키워드 포함 변수 표준화 합산 → 상위 30% = 위기
    feats = pick_features(df)
    cand = [c for c in feats if any(k in str(c).lower() for k in RISKY_WORDS)]
    if not cand:
        cand = feats[:min(10, len(feats))]
    zs = []
    for c in cand:
        v = pd.to_numeric(df[c], errors="coerce")
        mu, sd = v.mean(), v.std(ddof=0)
        if sd and not np.isnan(sd):
            zs.append((v - mu) / sd)
    if not zs:
        raise RuntimeError("타깃 생성 실패: 적절한 점수/이진/프록시 컬럼을 찾지 못함.")
    risk = np.nansum(np.vstack(zs), axis=0)
    thr = np.nanpercentile(risk, int(DERIVED_RISK_TOP*100))
    y = (risk >= thr).astype(int)
    return pd.Series(y), f"derived_proxy>=p{int(DERIVED_RISK_TOP*100)}({thr:.4f})", None, None, float(thr)

def main():
    df = load_csv(CSV_PATH)
    n_rows, n_cols = df.shape

    y, y_src, bin_col, score_col, thr_used = build_target(df)
    keep = ~pd.isna(y)
    df, y = df.loc[keep].copy(), y.loc[keep].astype(int)

    # X 선택
    exclude = set([c for c in [bin_col, score_col] if c is not None])
    X_cols = pick_features(df, exclude=exclude)
    X = df[X_cols].copy()

    # 결측: 수치형 중앙값 대체
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(X[c].median())

    # 서브샘플(시간/자원 보호)
    if MAX_ROWS_FOR_SPEED and len(X) > MAX_ROWS_FOR_SPEED:
        rng = np.random.default_rng(RANDOM_STATE)
        idx = rng.choice(len(X), size=MAX_ROWS_FOR_SPEED, replace=False)
        X, y = X.iloc[idx], y.iloc[idx]

    # 2) split & scaling
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, MinMaxScaler
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    std, mm = StandardScaler(), MinMaxScaler()
    Xtr_std, Xte_std = std.fit_transform(X_tr), std.transform(X_te)
    _ = mm.fit_transform(X_tr)  # 필요 시 사용

    # 3) GridSearch (RF, SVM)
    from sklearn.model_selection import GridSearchCV, StratifiedKFold
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

    rf_params = {
        "n_estimators": [150],
        "max_depth": [None, 10],
        "min_samples_split": [2],
        "min_samples_leaf": [1],
        "max_features": ["sqrt"],
        "class_weight": ["balanced"],
    }
    svm_params = {
        "C": [1],
        "gamma": ["scale"],
        "kernel": ["rbf"],
        "class_weight": ["balanced"],
    }

    rf_gs = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE),
                         rf_params, scoring="f1", cv=cv, n_jobs=1, verbose=0)
    rf_gs.fit(Xtr_std, y_tr)
    rf_best = rf_gs.best_estimator_

    svm_gs = GridSearchCV(SVC(probability=True, random_state=RANDOM_STATE),
                          svm_params, scoring="f1", cv=cv, n_jobs=1, verbose=0)
    svm_best = svm_gs.fit(Xtr_std, y_tr).best_estimator_

    # 3) + 4) 모델 학습 & 평가
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        "RandomForest(bestGS)": rf_best,
        "SVM(bestGS)": svm_best,
    }

    # XGBoost(설치돼 있으면 자동 사용)
    try:
        from xgboost import XGBClassifier
        pos_w = (len(y_tr) - y_tr.sum()) / (y_tr.sum() + 1e-9)
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.07,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=RANDOM_STATE, eval_metric="logloss",
            scale_pos_weight=float(pos_w)
        )
    except Exception:
        pass

    results = []
    for name, clf in models.items():
        clf.fit(Xtr_std, y_tr)
        y_pred = clf.predict(Xte_std)
        # AUC
        try:
            y_proba = clf.predict_proba(Xte_std)[:,1]
            roc = roc_auc_score(y_te, y_proba)
        except Exception:
            try:
                y_score = clf.decision_function(Xte_std)
                roc = roc_auc_score(y_te, y_score)
            except Exception:
                roc = np.nan
        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_te, y_pred),
            "Precision": precision_score(y_te, y_pred, zero_division=0),
            "Recall": recall_score(y_te, y_pred, zero_division=0),
            "F1": f1_score(y_te, y_pred, zero_division=0),
            "ROC_AUC": roc
        })

    metrics_df = pd.DataFrame(results).sort_values(
        by=["Recall","F1","ROC_AUC"], ascending=False
    ).reset_index(drop=True)

    summary_df = pd.DataFrame([{
        "n_rows_loaded": int(df.shape[0]),
        "n_cols_loaded": int(df.shape[1]),
        "n_samples_used": int(len(X)),
        "n_features_used": int(len(X.columns)),
        "target_source": y_src,
        "binary_target_col": bin_col,
        "score_target_col": score_col,
        "threshold_used": thr_used
    }])

    # 저장
    metrics_df.to_csv(OUT_METRICS, index=False)
    summary_df.to_csv(OUT_SUMMARY, index=False)
    pd.DataFrame({"feature": list(X.columns)}).to_csv(OUT_FEATURES, index=False)

    print("=== 모델 성능(Recall 우선 정렬) ===")
    print(metrics_df)
    print("\n=== 데이터/타깃 요약 ===")
    print(summary_df)
    print("\n저장 완료:")
    print(f"- {OUT_METRICS}")
    print(f"- {OUT_SUMMARY}")
    print(f"- {OUT_FEATURES}")

if __name__ == "__main__":
    main()